# 🔗 Gabungkan Semua Data GPS → 1 File Parquet

Notebook ini menggabungkan **9 file parquet bulanan** (Oktober 2021 – Juni 2022)
menjadi **1 file parquet gabungan** yang di-sort berdasarkan `maid` dan `timestamp`.

> **⚡ Pendekatan: DuckDB dengan Disk Spilling**
>
> Semua operasi (read, union, sort, write) dilakukan via DuckDB — tidak ada data
> yang di-load ke RAM via pandas. DuckDB akan spill ke disk jika RAM tidak cukup.
> Total data ~310 juta baris digabung tanpa risiko OOM.

**Langkah:**
1. Inventarisasi file parquet bulanan
2. Verifikasi schema setiap file
3. Gabungkan semua file → sort by `maid, timestamp` → tulis ke 1 file parquet
4. Validasi output

---
## 1 · Setup & Konfigurasi

In [1]:
import os
import time
from pathlib import Path

import duckdb

from shared import get_con, DATA_FILES

# ── Konfigurasi ──────────────────────────────────────────────────────────────
BASE_DIR = os.getcwd()
GPS_PARQUET_DIR = os.path.join(BASE_DIR, "DataGPS_parquet")
OUTPUT_FILE = os.path.join(GPS_PARQUET_DIR, "all_gps_data.parquet")

# DuckDB connection dengan disk spilling
con = get_con(BASE_DIR)

print("⚙️  DuckDB Configuration:")
for setting in ["memory_limit", "threads", "temp_directory"]:
    val = con.execute(f"SELECT current_setting('{setting}')").fetchone()[0]
    print(f"   {setting}: {val}")

print(f"\n📁 Sumber : {GPS_PARQUET_DIR}/")
print(f"📁 Output : {OUTPUT_FILE}")

⚙️  DuckDB Configuration:
   memory_limit: 3.7 GiB
   threads: 4
   temp_directory: /home/repal/Github/skripsi/./.duckdb_tmp

📁 Sumber : /home/repal/Github/skripsi/DataGPS_parquet/
📁 Output : /home/repal/Github/skripsi/DataGPS_parquet/all_gps_data.parquet


---
## 2 · Inventarisasi File Parquet Bulanan

In [2]:
print("=" * 70)
print("INVENTARISASI FILE PARQUET BULANAN")
print("=" * 70)

parquet_files = []
total_source_bytes = 0

for month_label, filename in DATA_FILES:
    fpath = os.path.join(GPS_PARQUET_DIR, filename)
    
    if not os.path.exists(fpath):
        print(f"  ❌ {filename} — TIDAK DITEMUKAN")
        continue
    
    fsize = os.path.getsize(fpath)
    total_source_bytes += fsize
    
    # Hitung jumlah baris via DuckDB (tanpa load ke RAM)
    row_count = con.execute(
        "SELECT COUNT(*) FROM read_parquet($1)", [fpath]
    ).fetchone()[0]
    
    parquet_files.append({
        "label": month_label,
        "filename": filename,
        "path": fpath,
        "size_bytes": fsize,
        "row_count": row_count,
    })
    
    print(f"  ✅ {filename:<35} {fsize / 1e6:>8.1f} MB   {row_count:>14,} baris")

print(f"\n{'─' * 70}")
print(f"  TOTAL: {len(parquet_files)} file"
      f"   {total_source_bytes / 1e6:>8.1f} MB"
      f"   {sum(f['row_count'] for f in parquet_files):>14,} baris")

INVENTARISASI FILE PARQUET BULANAN
  ✅ 2021_10_Oktober.parquet                 47.1 MB       12,123,709 baris
  ✅ 2021_11_November.parquet               355.3 MB      119,072,431 baris
  ✅ 2021_12_Desember.parquet               202.5 MB       45,950,349 baris
  ✅ 2022_01_Januari.parquet                145.2 MB       22,488,993 baris
  ✅ 2022_02_Februari.parquet               237.0 MB       39,601,777 baris
  ✅ 2022_03_Maret.parquet                   90.0 MB       26,837,642 baris
  ✅ 2022_04_April.parquet                    9.7 MB        1,048,571 baris
  ✅ 2022_05_Mei.parquet                    214.0 MB       31,075,618 baris
  ✅ 2022_06_Juni.parquet                    73.7 MB       12,246,261 baris

──────────────────────────────────────────────────────────────────────
  TOTAL: 9 file     1374.4 MB      310,445,351 baris


---
## 3 · Verifikasi Schema

In [5]:
print("=" * 70)
print("VERIFIKASI SCHEMA — Pastikan semua file konsisten")
print("=" * 70)

all_schemas_ok = True

for pf in parquet_files:
    schema = con.execute(f"""
        SELECT column_name, column_type
        FROM (DESCRIBE SELECT * FROM read_parquet('{pf['path']}'))
    """).df()
    
    expected = [("maid", "VARCHAR"), ("latitude", "DOUBLE"), ("longitude", "DOUBLE"), ("timestamp", "BIGINT")]
    actual = list(zip(schema["column_name"], schema["column_type"]))
    
    match = actual == expected
    status = "✅" if match else "❌"
    all_schemas_ok = all_schemas_ok and match
    
    print(f"  {status} {pf['label']:<25} → {actual}")

if all_schemas_ok:
    print(f"\n  ✅ Semua {len(parquet_files)} file memiliki schema yang konsisten!")
else:
    print("\n  ❌ PERINGATAN: Ada file dengan schema berbeda!")

VERIFIKASI SCHEMA — Pastikan semua file konsisten
  ✅ 2021_10_Oktober           → [('maid', 'VARCHAR'), ('latitude', 'DOUBLE'), ('longitude', 'DOUBLE'), ('timestamp', 'BIGINT')]
  ✅ 2021_11_November          → [('maid', 'VARCHAR'), ('latitude', 'DOUBLE'), ('longitude', 'DOUBLE'), ('timestamp', 'BIGINT')]
  ✅ 2021_12_Desember          → [('maid', 'VARCHAR'), ('latitude', 'DOUBLE'), ('longitude', 'DOUBLE'), ('timestamp', 'BIGINT')]
  ✅ 2022_01_Januari           → [('maid', 'VARCHAR'), ('latitude', 'DOUBLE'), ('longitude', 'DOUBLE'), ('timestamp', 'BIGINT')]
  ✅ 2022_02_Februari          → [('maid', 'VARCHAR'), ('latitude', 'DOUBLE'), ('longitude', 'DOUBLE'), ('timestamp', 'BIGINT')]
  ✅ 2022_03_Maret             → [('maid', 'VARCHAR'), ('latitude', 'DOUBLE'), ('longitude', 'DOUBLE'), ('timestamp', 'BIGINT')]
  ✅ 2022_04_April             → [('maid', 'VARCHAR'), ('latitude', 'DOUBLE'), ('longitude', 'DOUBLE'), ('timestamp', 'BIGINT')]
  ✅ 2022_05_Mei               → [('maid', 'VARCHAR'), 

---
## 4 · Gabungkan Semua File → Sort → Tulis Parquet

DuckDB akan:
1. Membaca semua 9 file parquet secara lazy (tidak load ke RAM)
2. Melakukan `UNION ALL` (penggabungan)
3. Sort berdasarkan `maid, timestamp` (out-of-core, spill ke disk jika perlu)
4. Menulis hasilnya ke 1 file parquet dengan kompresi ZSTD

In [6]:
if os.path.exists(OUTPUT_FILE):
    existing_size = os.path.getsize(OUTPUT_FILE)
    print(f"⚠️  File output sudah ada: {OUTPUT_FILE}")
    print(f"   Ukuran: {existing_size / 1e6:.1f} MB")
    print(f"   Jika ingin menggabungkan ulang, hapus file ini terlebih dahulu.")
    print(f"   Melewati proses penggabungan...")
    SKIP_MERGE = True
else:
    SKIP_MERGE = False

In [7]:
if not SKIP_MERGE:
    print("=" * 70)
    print("MENGGABUNGKAN SEMUA DATA GPS")
    print("=" * 70)
    
    # Bangun daftar path untuk DuckDB glob
    all_paths = [pf["path"] for pf in parquet_files]
    total_rows_expected = sum(pf["row_count"] for pf in parquet_files)
    
    print(f"  📥 Input  : {len(all_paths)} file parquet")
    print(f"  📊 Total  : {total_rows_expected:,} baris")
    print(f"  📤 Output : {OUTPUT_FILE}")
    print(f"  🔀 Sort   : maid, timestamp (ascending)")
    print(f"  📦 Kompresi: ZSTD")
    print()
    
    # Buat list file sebagai string SQL
    file_list_sql = ", ".join(f"'{p}'" for p in all_paths)
    
    t0 = time.time()
    print("  ⏳ Proses dimulai...", flush=True)
    
    # Gunakan COPY ... TO untuk menulis langsung ke parquet
    # DuckDB akan melakukan sort out-of-core (disk spilling)
    con.execute(f"""
        COPY (
            SELECT maid, latitude, longitude, timestamp
            FROM read_parquet([{file_list_sql}])
            ORDER BY maid, timestamp
        )
        TO '{OUTPUT_FILE}'
        (FORMAT PARQUET, COMPRESSION ZSTD, ROW_GROUP_SIZE 122880)
    """)
    
    elapsed = time.time() - t0
    output_size = os.path.getsize(OUTPUT_FILE)
    
    print(f"\n  ✅ Selesai dalam {elapsed:.1f} detik ({elapsed / 60:.1f} menit)")
    print(f"  📁 Output : {OUTPUT_FILE}")
    print(f"  📏 Ukuran : {output_size / 1e6:.1f} MB ({output_size / 1e9:.2f} GB)")
    print(f"  📊 Kompresi: {total_source_bytes / output_size:.2f}x dari sumber")
else:
    print("  ⏭️  Proses penggabungan dilewati (file sudah ada)")

MENGGABUNGKAN SEMUA DATA GPS
  📥 Input  : 9 file parquet
  📊 Total  : 310,445,351 baris
  📤 Output : /home/repal/Github/skripsi/DataGPS_parquet/all_gps_data.parquet
  🔀 Sort   : maid, timestamp (ascending)
  📦 Kompresi: ZSTD

  ⏳ Proses dimulai...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


  ✅ Selesai dalam 309.6 detik (5.2 menit)
  📁 Output : /home/repal/Github/skripsi/DataGPS_parquet/all_gps_data.parquet
  📏 Ukuran : 1265.5 MB (1.27 GB)
  📊 Kompresi: 1.09x dari sumber


---
## 5 · Validasi Output

In [9]:
print("=" * 70)
print("VALIDASI OUTPUT")
print("=" * 70)

if os.path.exists(OUTPUT_FILE):
    output_size = os.path.getsize(OUTPUT_FILE)
    
    # Hitung total baris output
    output_rows = con.execute(
        "SELECT COUNT(*) FROM read_parquet($1)", [OUTPUT_FILE]
    ).fetchone()[0]
    
    expected_rows = sum(pf["row_count"] for pf in parquet_files)
    rows_match = output_rows == expected_rows
    
    print(f"  📏 Ukuran file    : {output_size / 1e6:.1f} MB ({output_size / 1e9:.2f} GB)")
    print(f"  📊 Baris output   : {output_rows:,}")
    print(f"  📊 Baris expected : {expected_rows:,}")
    print(f"  {'✅' if rows_match else '❌'} Row count {'cocok' if rows_match else 'TIDAK COCOK'}!")
    
    # Cek schema output
    schema_out = con.execute(f"""
        SELECT column_name, column_type 
        FROM (DESCRIBE SELECT * FROM read_parquet('{OUTPUT_FILE}'))
    """).df()
    
    print(f"\n  Schema output:")
    for _, row in schema_out.iterrows():
        print(f"    {row['column_name']:<15} {row['column_type']}")
    
    # Cek jumlah distinct maid
    distinct_maid = con.execute(
        "SELECT COUNT(DISTINCT maid) FROM read_parquet($1)", [OUTPUT_FILE]
    ).fetchone()[0]
    print(f"\n  📍 Distinct MAID  : {distinct_maid:,}")
    
    # Cek rentang timestamp
    ts_range = con.execute(f"""
        SELECT
            STRFTIME(TO_TIMESTAMP(MIN(timestamp)), '%Y-%m-%d %H:%M:%S') AS ts_min,
            STRFTIME(TO_TIMESTAMP(MAX(timestamp)), '%Y-%m-%d %H:%M:%S') AS ts_max
        FROM read_parquet('{OUTPUT_FILE}')
    """).df()
    print(f"  📅 Timestamp min  : {ts_range['ts_min'].iloc[0]}")
    print(f"  📅 Timestamp max  : {ts_range['ts_max'].iloc[0]}")
    
    # Preview data
    print(f"\n  Preview (5 baris pertama):")
    preview = con.execute(
        "SELECT * FROM read_parquet($1) LIMIT 5", [OUTPUT_FILE]
    ).df()
    print(preview.to_string(index=False))
    
    # Verifikasi sorting
    sort_check = con.execute(f"""
        WITH numbered AS (
            SELECT maid, timestamp,
                   LAG(maid) OVER () AS prev_maid,
                   LAG(timestamp) OVER () AS prev_ts
            FROM read_parquet('{OUTPUT_FILE}')
            LIMIT 1000000
        )
        SELECT COUNT(*) AS violations
        FROM numbered
        WHERE prev_maid IS NOT NULL
          AND (maid < prev_maid 
               OR (maid = prev_maid AND timestamp < prev_ts))
    """).fetchone()[0]
    
    print(f"\n  🔀 Sort violations (1M baris pertama): {sort_check:,}")
    print(f"  {'✅' if sort_check == 0 else '⚠️'} Data {'tersortir dengan benar' if sort_check == 0 else 'TIDAK tersortir!'}")

else:
    print("  ❌ File output tidak ditemukan!")

VALIDASI OUTPUT
  📏 Ukuran file    : 1265.5 MB (1.27 GB)
  📊 Baris output   : 310,445,351
  📊 Baris expected : 310,445,351
  ✅ Row count cocok!

  Schema output:
    maid            VARCHAR
    latitude        DOUBLE
    longitude       DOUBLE
    timestamp       BIGINT

  📍 Distinct MAID  : 4,281,537
  📅 Timestamp min  : 2021-10-23 07:00:00
  📅 Timestamp max  : 2022-06-08 02:27:45

  Preview (5 baris pertama):
                                maid  latitude  longitude  timestamp
0042c956-6867-4c68-acef-8aef058b9294      -8.0 110.400002 1639290507
0042c956-6867-4c68-acef-8aef058b9294      -8.0 110.400002 1639290523
0042c956-6867-4c68-acef-8aef058b9294      -8.0 110.400002 1639290523
0042c956-6867-4c68-acef-8aef058b9294      -8.0 110.400002 1639290545
0042c956-6867-4c68-acef-8aef058b9294      -8.0 110.400002 1639290546

  🔀 Sort violations (1M baris pertama): 0
  ✅ Data tersortir dengan benar


---
## 6 · Ringkasan

In [10]:
print("=" * 70)
print("                     📋 RINGKASAN PENGGABUNGAN")
print("                     Data GPS Oktober 2021 – Juni 2022")
print("=" * 70)

if os.path.exists(OUTPUT_FILE):
    print(f"""
  📥 Sumber         : {len(parquet_files)} file parquet bulanan
  📏 Total sumber   : {total_source_bytes / 1e6:.1f} MB ({total_source_bytes / 1e9:.2f} GB)
  📤 Output         : {os.path.basename(OUTPUT_FILE)}
  📏 Ukuran output  : {output_size / 1e6:.1f} MB ({output_size / 1e9:.2f} GB)
  📊 Total baris    : {output_rows:,}
  📍 Distinct MAID  : {distinct_maid:,}
  📅 Rentang waktu  : {ts_range['ts_min'].iloc[0]} s/d {ts_range['ts_max'].iloc[0]}
  🔀 Sorting        : maid, timestamp (ascending)
  📦 Kompresi       : ZSTD
""")

print("=" * 70)
print("  ✅ Selesai!")
print("=" * 70)

                     📋 RINGKASAN PENGGABUNGAN
                     Data GPS Oktober 2021 – Juni 2022

  📥 Sumber         : 9 file parquet bulanan
  📏 Total sumber   : 1374.4 MB (1.37 GB)
  📤 Output         : all_gps_data.parquet
  📏 Ukuran output  : 1265.5 MB (1.27 GB)
  📊 Total baris    : 310,445,351
  📍 Distinct MAID  : 4,281,537
  📅 Rentang waktu  : 2021-10-23 07:00:00 s/d 2022-06-08 02:27:45
  🔀 Sorting        : maid, timestamp (ascending)
  📦 Kompresi       : ZSTD

  ✅ Selesai!


In [11]:
# Tutup koneksi DuckDB
con.close()
print("✅ Koneksi DuckDB ditutup")

✅ Koneksi DuckDB ditutup
